&nbsp;

# Metadata Integration and Visualization

- 様々な公共データベースから取得したデータを統合する

In [1]:
import os
import datetime
import pathlib
# import numpy as np
# import pandas as pd
import polars as pl
import plotly.express as px

from IPython.display import display

pl.Config.set_tbl_rows(6)
pl.Config.set_fmt_str_lengths(500)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

# create directory
pathlib.Path(f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 0. HN-score Table Integration

- 収集した全てのデータから算出した HN-score と、`stranded` と判定したデータのみから算出した HN-score を統合する

In [3]:
count_all = pl.read_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/HNscore_gencode_human_261001_HN2_transcript_all.tsv",
    separator="\t"
).with_columns(
    # Add temporary key for RNAcentral join
    pl.col("transcript_id").str.replace(r"\.\d+$", "").alias("transcript_id_no_version")
)

display(count_all)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UPcount+1 / DOWNcount+1 2,log2(UPcount+1 / DOWNcount+1) 2,transcript_id_no_version
str,i64,i64,i64,i64,f64,f64,str
"""ENST00000309758.6""",352,17,37,335,19.611111,4.293599,"""ENST00000309758"""
"""ENST00000679223.2""",338,8,60,330,37.666667,5.235216,"""ENST00000679223"""
"""ENST00000450744.1""",335,13,58,322,24.0,4.584963,"""ENST00000450744"""
…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,0.045113,-4.47032,"""ENST00000549545"""
"""ENST00000426094.5""",11,265,130,-254,0.045113,-4.47032,"""ENST00000426094"""
"""ENST00000560179.2""",11,272,123,-261,0.043956,-4.507795,"""ENST00000560179"""


In [4]:
count_stranded = pl.read_csv(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/HNscore_gencode_human_261001_HN2_transcript_all_stranded.tsv",
    separator="\t",
    n_threads=16
).with_columns(
    # Add temporary key for RNAcentral join
    pl.col("transcript_id").str.replace(r"\.\d+$", "").alias("transcript_id_no_version")
)

display(count_stranded) 

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UPcount+1 / DOWNcount+1 2,log2(UPcount+1 / DOWNcount+1) 2,transcript_id_no_version
str,i64,i64,i64,i64,f64,f64,str
"""ENST00000309758.6""",232,4,15,228,46.6,5.542258,"""ENST00000309758"""
"""ENST00000679223.2""",222,1,28,221,111.5,6.8009,"""ENST00000679223"""
"""ENST00000450744.1""",223,4,24,219,44.8,5.485427,"""ENST00000450744"""
…,…,…,…,…,…,…,…
"""ENST00000299238.7""",8,181,62,-173,0.049451,-4.33787,"""ENST00000299238"""
"""ENST00000638929.1""",8,183,60,-175,0.048913,-4.353637,"""ENST00000638929"""
"""ENST00000560179.2""",5,181,65,-176,0.032967,-4.922832,"""ENST00000560179"""


In [5]:
count_integration = count_all.join(
    count_stranded,
    on=["transcript_id", "transcript_id_no_version"],
    how="inner",
    suffix="_stranded"
).select(
    "transcript_id",
    "UP2",
    "DOWN2",
    "UNCHANGED2",
    "HN2",
    "UPcount+1 / DOWNcount+1 2",
    "log2(UPcount+1 / DOWNcount+1) 2",
    "UP2_stranded",
    "DOWN2_stranded",
    "UNCHANGED2_stranded",
    "HN2_stranded",
    "UPcount+1 / DOWNcount+1 2_stranded",
    "log2(UPcount+1 / DOWNcount+1) 2_stranded",
    "transcript_id_no_version"
)

count_integration.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/HNscore_gencode_human_{Execute_date}_HN2_transcript_all_integration.tsv",
    separator="\t"
)

display(count_integration)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UPcount+1 / DOWNcount+1 2,log2(UPcount+1 / DOWNcount+1) 2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,UPcount+1 / DOWNcount+1 2_stranded,log2(UPcount+1 / DOWNcount+1) 2_stranded,transcript_id_no_version
str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,str
"""ENST00000309758.6""",352,17,37,335,19.611111,4.293599,232,4,15,228,46.6,5.542258,"""ENST00000309758"""
"""ENST00000679223.2""",338,8,60,330,37.666667,5.235216,222,1,28,221,111.5,6.8009,"""ENST00000679223"""
"""ENST00000450744.1""",335,13,58,322,24.0,4.584963,223,4,24,219,44.8,5.485427,"""ENST00000450744"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000299238.7""",9,245,152,-236,0.04065,-4.620586,8,181,62,-173,0.049451,-4.33787,"""ENST00000299238"""
"""ENST00000638929.1""",18,259,129,-241,0.073077,-3.77444,8,183,60,-175,0.048913,-4.353637,"""ENST00000638929"""
"""ENST00000560179.2""",11,272,123,-261,0.043956,-4.507795,5,181,65,-176,0.032967,-4.922832,"""ENST00000560179"""


&nbsp;

&nbsp;

&nbsp;

# Add Meta-data

## 1. GENCODE metadata

&nbsp;

&nbsp;

## 1-1. transcript - gene correspondence

In [6]:
# using ../Tools/06_pytximport_tx2gene.cwl to get tx2gene from gencode.v49.chr_patch_hapl_scaff.annotation.gtf file

tx2gene = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode_v49_tx2gene_chr_patch_hapl_scaff_v260909.tsv",
    separator="\t",
    n_threads=16
)

pl.Config.set_tbl_rows(4)
display(tx2gene)

transcript_id,gene_id
str,str
"""ENST00000832824.1""","""ENSG00000290825.2"""
"""ENST00000832825.1""","""ENSG00000290825.2"""
…,…
"""ENST00000751368.1""","""ENSG00000297844.1"""
"""ENST00000839853.1""","""ENSG00000309258.1"""


&nbsp;

&nbsp;

&nbsp;

## 1-2. transcript type retrieved from [gencode.v49.chr_patch_hapl_scaff.annotation.gtf](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf.gz)

- (2026-09-04) strand 情報をここで統合することにした
- (2026-09-08) [tags](https://www.gencodegenes.org/pages/tags.html) のデータもGTFファイルから取得することにした

In [7]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
tx2type = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == 'transcript'
).with_columns(
    # Unique keys: extract the first (and only) quoted value
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    pl.col("attributes").str.extract(r'transcript_type "([^"]+)"', 1).alias("transcript_type"),
    # tag can appear multiple times; extract_all returns full matches like tag "basic"
    pl.col("attributes")
        .str.extract_all(r'tag "[^"]+"')
        .list.eval(pl.element().str.extract(r'"([^"]+)"', 1))
        .alias("tags")
).select([
    pl.col("transcript_id"),
    pl.col("transcript_type"),
    pl.col("gene_id"),
    pl.col("tags"),
    pl.col("strand")
]).with_columns(
    pl.col("tags").list.n_unique().alias("number_of_tags")
).unique(
    subset=["transcript_id", "transcript_type", "gene_id", "tags", "number_of_tags", "strand"]
).join(
    # きちんと transcript_id と gene_id の対応が保持されているかを確認する
    tx2gene,
    on=["transcript_id", "gene_id"],
    how="inner"
)

display(tx2type)

transcript_id,transcript_type,gene_id,tags,strand,number_of_tags
str,str,str,list[str],str,u32
"""ENST00000832824.1""","""lncRNA""","""ENSG00000290825.2""","[""TAGENE""]","""+""",1
"""ENST00000832825.1""","""lncRNA""","""ENSG00000290825.2""","[""TAGENE""]","""+""",1
…,…,…,…,…,…
"""ENST00000751368.1""","""lncRNA""","""ENSG00000297844.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4
"""ENST00000839853.1""","""lncRNA""","""ENSG00000309258.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1-3. GENCODE promoter windows (v48)

- ~~(2026-09-04) このメタデータには、GENCODE の promoter 情報は統合しないことにした~~
- (2026-09-08) 後続の解析である `fanta.bio`における promoter の証拠としても使用できるため、統合する
- (2026-09-08) `fanta.bio`の BEDファイルと後で `bedtools intersect` しておくので、BEDファイルを作成しておく
- → `gffread` では、どうやら [gencode.v48.promoter_windows.gff3.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.promoter_windows.gff3.gz) の feature の タイプ名のところが Sequence Ontology Feature Annotation (SOFA) に該当しない？ ため、うまく変換できないので、データフレームとして操作して変換する
- [../02_fantabio/02_5_ensembl_promoter_intersect.ipynb](../02_fantabio/02_5_ensembl_promoter_intersect.ipynb)に処理を移動
- https://www.gencodegenes.org/pages/promoter_windows/
- [gencode.v48.promoter_windows.gff3.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.promoter_windows.gff3.gz)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1−4. annotation remark 

- [ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Annotation_remark.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Annotation_remark.gz)

In [8]:
annotation_remark = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.Annotation_remark.gz",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "transcript_id",
        "annotation_remark"
    ],
    quote_char=None,
    infer_schema=False,
    ignore_errors=True
).sort(
    by="transcript_id",
    descending=False
).group_by(
    "transcript_id"
).agg(
    pl.col("annotation_remark").alias("annotation_remarks"),
    pl.col("annotation_remark").count().fill_null(0).alias("annotation_remark_count")
)

display(annotation_remark)

transcript_id,annotation_remarks,annotation_remark_count
str,list[str],u32
"""ENST00000920101.1""","[""LRGASP""]",1
"""ENST00000783444.1""","[""CLS3 project""]",1
…,…,…
"""ENST00000834112.1""","[""CLS3 project""]",1
"""ENST00000885842.1""","[""DanaFarber""]",1


In [9]:
annotation_remark_list = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.Annotation_remark.gz",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "transcript_id",
        "annotation_remark"
    ],
    quote_char=None,
    infer_schema=False,
    ignore_errors=True
).sort(
    by="transcript_id",
    descending=False
).select(
    "annotation_remark"
).group_by(
    "annotation_remark"
).agg(
    pl.col("annotation_remark").count().alias("annotation_remark_count")
).sort(
    by="annotation_remark_count",
    descending=True
)

annotation_remark_list.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/gencode.v49.metadata.Annotation_remark_list.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(4)
display(annotation_remark_list)

annotation_remark,annotation_remark_count
str,u32
"""CLS3 project""",146072
"""DanaFarber""",60225
…,…
"""H2B histone family, member K (H2B/k)""",1
"""IGLV3-10*01, V2-7""",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1−5. Transcript supporting feature

- [ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Transcript_supporting_feature.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Transcript_supporting_feature.gz)

In [10]:
# # Piece of evidence used in the annotation of the transcript (GENCODE v49)
# transcript_support_feature = pl.read_csv(
#     "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.Transcript_supporting_feature.gz",
#     separator="\t",
#     has_header=False,
#     n_threads=16,
#     new_columns=[
#         "transcript_id",
#         "transcript_support_id",
#         "transcript_annotation_evidence"
#     ]
# ).group_by(
#     "transcript_id"
# ).agg(
#     pl.col("transcript_support_id").alias("transcript_support_ids"),
#     pl.col("transcript_annotation_evidence").alias("transcript_annotation_evidences")
# )

# pl.Config.set_tbl_rows(4)
# display(transcript_support_feature)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1−6. Gene Source

- [ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Gene_source.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.Gene_source.gz)

In [11]:
gene_source = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.Gene_source.gz",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "gene_id",
        "gene_source"
    ]
)

display(gene_source)

gene_id,gene_source
str,str
"""ENSG00000142611.17""","""ensembl_havana_gene_homo_sapiens"""
"""ENSG00000284616.1""","""havana_homo_sapiens"""
…,…
"""ENSG00000297844.1""","""havana_tagene_homo_sapiens"""
"""ENSG00000309258.1""","""havana_tagene_homo_sapiens"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1−7. HGNC symbol

- [ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.HGNC.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.HGNC.gz)

In [12]:
tx2symbol = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.HGNC.gz",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "transcript_id",
        "gene_symbol",
        "HGNC_ID"
    ]
)

display(tx2symbol)

transcript_id,gene_symbol,HGNC_ID
str,str,str
"""ENST00000511072.5""","""PRDM16""","""HGNC:14000"""
"""ENST00000607632.1""","""PRDM16""","""HGNC:14000"""
…,…,…
"""ENST00000611446.1""","""RNA5-8SN5""","""HGNC:53533"""
"""ENST00000616292.1""","""RNA5-8SN4""","""HGNC:53529"""


&nbsp;

&nbsp;

&nbsp;

## 1−8. poly-A feature

- 詳細な場所ではなく、poly-A position が GENCODE でアノテーションされているflagとして使用
- [ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.PolyA_feature.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.metadata.PolyA_feature.gz)

In [13]:
polyA_feature = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v49.metadata.PolyA_feature.gz",
    separator="\t",
    has_header=False,
    n_threads=16
).select(
    "column_1",
    "column_8"
).rename(
    {
        "column_1": "transcript_id",
        "column_8": "polyA_feature"
    }
).unique().group_by(
    "transcript_id"
).agg(
    pl.col("polyA_feature").alias("polyA_features")
)

display(polyA_feature)

transcript_id,polyA_features
str,list[str]
"""ENST00000484161.5""","[""polya_site"", ""polya_signal""]"
"""ENST00000955461.1""","[""polya_signal"", ""polya_site""]"
…,…
"""ENST00000360506.6""","[""polya_signal"", ""polya_site""]"
"""ENST00000895099.1""","[""polya_site""]"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. RNACentral ID correspondence (for lncRNA information)

- RNACentral release 27 からダウンロード (2026-10-01 access)
- https://ftp.ebi.ac.uk/pub/databases/RNAcentral/releases/27.0/id_mapping/database_mappings/ensembl_gencode.tsv
- transcript id は、バージョンがない状態でなければ結合できないことに注意！

In [14]:
# caution: transcript id is not versioned !
# curl -O https://ftp.ebi.ac.uk/pub/databases/RNAcentral/releases/27.0/id_mapping/database_mappings/ensembl_gencode.tsv (2026-10-01)
rnacentral_id = pl.read_csv(
    "../../Data/Database/RNACentral/release27/database_mapping/ensembl_gencode.tsv",
    separator="\t",
    has_header=False,
    n_threads=16
).rename(
    {
        "column_1": "RNACentral_ID",
        "column_2": "corrrespondence",
        "column_3": "transcript_id_no_version",
        "column_4": "taxonomy_id",
        "column_5": "transcript_type",
        "column_6": "gene_id"
    }
).filter(
    pl.col("taxonomy_id") == 9606
).select(
    "RNACentral_ID",
    "corrrespondence",
    "transcript_id_no_version"
).unique()

display(rnacentral_id)

RNACentral_ID,corrrespondence,transcript_id_no_version
str,str,str
"""URS0002A322D2""","""ENSEMBL_GENCODE""","""ENST00000842269"""
"""URS0002A6B04F""","""ENSEMBL_GENCODE""","""ENST00000570344"""
…,…,…
"""URS0002A50320""","""ENSEMBL_GENCODE""","""ENST00000725801"""
"""URS000055A804""","""ENSEMBL_GENCODE""","""ENST00000446022"""


&nbsp;

&nbsp;

## JOIN

- これまでに収集したメタデータを全て `join`
- `salmon` の index 作成時に 短すぎて 発現定量できていない transcript はここで落ちている

In [15]:
HNscore_add_metadata = count_integration.join(
    tx2symbol,
    on="transcript_id",
    how="left"
).join(
    tx2type,
    on="transcript_id",
    how="left"
).join(
    rnacentral_id,
    on="transcript_id_no_version", # caution: transcript id is not versioned !
    how="left"
).join(
    polyA_feature,
    on="transcript_id",
    how="left"
).join(
    gene_source,
    on="gene_id",
    how="left"
).drop(
    "UPcount+1 / DOWNcount+1 2",
    "UPcount+1 / DOWNcount+1 2_stranded",
    "log2(UPcount+1 / DOWNcount+1) 2",
    "log2(UPcount+1 / DOWNcount+1) 2_stranded"
).join(
    annotation_remark,
    on="transcript_id",
    how="left"
).with_columns([
    pl.col("annotation_remark_count").fill_null(0)
]).sort(
    by="HN2",
    descending=True
).drop(
    "transcript_id_no_version"
)

pl.Config.set_tbl_rows(20)
display(HNscore_add_metadata)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","[""alternative_5_UTR"", ""inferred_transcript_model"", … ""CCDS""]","""-""",5,null,null,"[""polya_site""]","""ensembl_havana_gene_homo_sapiens""","[""alternative 5' UTR"", ""inferred transcript model"", ""RNA-Seq supported only""]",3
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","[""non_canonical_TEC""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""non canonical TEC""]",1
"""ENST00000676231.2""",324,7,75,317,209,2,40,207,"""HSPB1""","""HGNC:5246""","""protein_coding""","""ENSG00000106211.11""","[""inferred_exon_combination"", ""RNA_Seq_supported_only"", ""basic""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""intropolis_filtered"", ""Snaptron_GTEX"", … ""RNA-Seq supported only""]",4
"""ENST00000401903.6""",334,17,55,317,210,4,37,206,"""ZFAND2A""","""HGNC:28073""","""protein_coding""","""ENSG00000178381.14""","[""not_organism_supported"", ""basic"", … ""CCDS""]","""-""",5,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""not organism-supported""]",1
"""ENST00000954442.1""",328,14,64,314,209,3,39,206,"""HSPD1""","""HGNC:5261""","""protein_coding""","""ENSG00000144381.19""","[""basic"", ""TAGENE"", … ""CCDS""]","""-""",4,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""ENCODE_tissue""]",1
"""ENST00000555662.1""",329,18,59,311,215,5,31,210,"""HSP90AA1""","""HGNC:5253""","""retained_intron""","""ENSG00000080824.20""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000850140.1""",317,6,83,311,207,2,42,205,null,null,"""lncRNA""","""ENSG00000310471.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A15385""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project"", ""overlaps pseudogene""]",2


In [16]:
threshold = 2

In [17]:
HNscore_add_metadata.write_parquet(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/HNscore_gencode_human_{Execute_date}_HN{threshold}_transcript_all_gencode_annotation.parquet"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Filtered Transcripts with no HN-score

In [18]:
transcript_with_no_HNscore = tx2gene.join(
    count_integration.select("transcript_id"),
    on="transcript_id",
    how="anti",
)

transcript_with_no_HNscore.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/transcript_with_no_HNscore.tsv",
    separator="\t"
)

display(transcript_with_no_HNscore)

transcript_id,gene_id
str,str
"""ENST00000612080.1""","""ENSG00000273874.1"""
"""ENST00000614007.1""","""ENSG00000278757.1"""
"""ENST00000332831.5""","""ENSG00000284662.2"""
"""ENST00000377898.4""","""ENSG00000173673.9"""
"""ENST00000384780.1""","""ENSG00000207511.1"""
"""ENST00000384782.1""","""ENSG00000207513.1"""
"""ENST00000849135.1""","""ENSG00000282740.2"""
"""ENST00000384659.1""","""ENSG00000207389.1"""
"""ENST00000384278.1""","""ENSG00000207005.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Assign the Original Flags

In [19]:
HNscore_add_metadata_with_score = HNscore_add_metadata.filter(
    pl.col(f"HN{threshold}").is_not_null()
).with_columns([
    pl.col("annotation_remarks").list.join(", ").alias("annotation_remarks"),
    pl.col("polyA_features").list.join(", ").alias("polyA_features"),
    pl.col("tags").list.join(", ").alias("tags")
]).with_columns( # For pie chart
    pl.when(
        pl.col("annotation_remarks").str.contains("CLS3 project") |
        pl.col("annotation_remarks").str.contains("Assembled from PacBio CLS3 reads")
    ).then(pl.lit("derived from CLS3 project"))
    .when(
        pl.col("annotation_remarks").str.contains("MANE_select")
    ).then(pl.lit("MANE_select"))
    .when(
        pl.col("annotation_remarks").str.contains("LRGASP")
    ).then(pl.lit("derived from LRGASP"))
    .when(
        pl.col("annotation_remarks").is_null()
    ).then(pl.lit("no annotation remark"))
    .otherwise(pl.lit("Other"))
    .alias("representative_annotation_remarks")
).with_columns(
    pl.when(
        (pl.col("transcript_type") == "protein_coding") 
    ).then(pl.lit("protein_coding"))

    .when( # For more information, see https://www.gencodegenes.org/pages/biotypes.html
        (pl.col("transcript_type") == "lncRNA") |
        (pl.col("transcript_type") == "bidirectional_promoter_lncRNA") |
        (pl.col("transcript_type") == "macro_lncRNA") |
        (pl.col("transcript_type") == "lincRNA")
    ).then(pl.lit("lncRNA"))
    .otherwise(pl.lit("others"))
    .alias("representative_transcript_type")
)

HNscore_add_metadata_with_score.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/HNscore_gencode_human_{Execute_date}_HN{threshold}_transcript_all_gencode_annotation.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(10)
# print(HNscore_add_metadata_with_score.select("transcript_type").unique())
display(HNscore_add_metadata_with_score)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,u32,str,str,str,str,str,u32,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","""basic, Ensembl_canonical, GENCODE_Primary, MANE_Select, appris_principal_1, CCDS""","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""MANE_select""",1,"""MANE_select""","""protein_coding"""
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","""alternative_5_UTR, inferred_transcript_model, RNA_Seq_supported_only, basic, CCDS""","""-""",5,null,null,"""polya_site""","""ensembl_havana_gene_homo_sapiens""","""alternative 5' UTR, inferred transcript model, RNA-Seq supported only""",3,"""Other""","""protein_coding"""
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","""basic""","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""protein_coding"""
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","""non_canonical_TEC""","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""non canonical TEC""",1,"""Other""","""others"""
"""ENST00000676231.2""",324,7,75,317,209,2,40,207,"""HSPB1""","""HGNC:5246""","""protein_coding""","""ENSG00000106211.11""","""inferred_exon_combination, RNA_Seq_supported_only, basic""","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""intropolis_filtered, Snaptron_GTEX, inferred exon combination, RNA-Seq supported only""",4,"""Other""","""protein_coding"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000497231.7""",15,266,125,-251,10,182,59,-172,"""ALDH7A1""","""HGNC:877""","""retained_intron""","""ENSG00000164904.19""","""""","""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000490762.5""",12,265,129,-253,10,174,67,-164,"""LUC7L""","""HGNC:6723""","""retained_intron""","""ENSG00000007392.18""","""""","""-""",0,null,null,"""polya_site, polya_signal""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000426094.5""",11,265,130,-254,11,165,75,-154,"""LUC7L""","""HGNC:6723""","""nonsense_mediated_decay""","""ENSG00000007392.18""","""""","""-""",0,null,null,"""polya_site, polya_signal""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Upregulated Gene Groups

- データが多すぎて`join`が難しいので、選抜してから行う

In [20]:
top5_genes_list = pl.read_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/HN{threshold}_top_5_percent_transcripts.tsv",
    separator="\t"
)

# Use semi join to filter rows where transcript_id exists in top1_genes_list
top5_genes = HNscore_add_metadata_with_score.join(
    top5_genes_list.select("transcript_id"),
    on="transcript_id",
    how="semi" # Returns rows from the left table that have a match in the right table.
)

top5_genes.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/HN{threshold}_top5p_transcripts_annotation_{Execute_date}.tsv",
    separator="\t"
)

display(top5_genes)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,u32,str,str,str,str,str,u32,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","""basic, Ensembl_canonical, GENCODE_Primary, MANE_Select, appris_principal_1, CCDS""","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""MANE_select""",1,"""MANE_select""","""protein_coding"""
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","""alternative_5_UTR, inferred_transcript_model, RNA_Seq_supported_only, basic, CCDS""","""-""",5,null,null,"""polya_site""","""ensembl_havana_gene_homo_sapiens""","""alternative 5' UTR, inferred transcript model, RNA-Seq supported only""",3,"""Other""","""protein_coding"""
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","""basic""","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""protein_coding"""
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","""non_canonical_TEC""","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""non canonical TEC""",1,"""Other""","""others"""
"""ENST00000676231.2""",324,7,75,317,209,2,40,207,"""HSPB1""","""HGNC:5246""","""protein_coding""","""ENSG00000106211.11""","""inferred_exon_combination, RNA_Seq_supported_only, basic""","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""intropolis_filtered, Snaptron_GTEX, inferred exon combination, RNA-Seq supported only""",4,"""Other""","""protein_coding"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000377777.6""",112,68,226,44,47,46,158,1,"""H2BC5""","""HGNC:4747""","""protein_coding""","""ENSG00000158373.10""","""basic, Ensembl_canonical, GENCODE_Primary, MANE_Select, appris_principal_1, CCDS""","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","""MANE_select, novel histone 2B family member, variant 2, alternatively spliced in 3' UTR""",3,"""MANE_select""","""protein_coding"""
"""ENST00000430496.1""",71,27,308,44,17,16,218,1,"""PRPS1P2""","""HGNC:9464""","""processed_pseudogene""","""ENSG00000232630.1""","""pseudo_consens, basic, Ensembl_canonical, GENCODE_Primary""","""+""",4,null,null,null,"""havana_homo_sapiens""","""phosphoribosyl pyrophosphate synthetase 1 (PRPS1) pseudogene""",1,"""Other""","""others"""
"""ENST00000480008.1""",114,70,222,44,53,53,145,0,"""PHF10""","""HGNC:18250""","""retained_intron""","""ENSG00000130024.16""","""""","""-""",0,null,null,"""polya_signal, polya_site""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""


&nbsp;

&nbsp;

## Downregulated Gene Group

In [21]:
bottom5_genes_list = pl.read_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/HN{threshold}_bottom_5_percent_transcripts.tsv",
    separator="\t"
)

# Use semi join to filter rows where transcript_id exists in top1_genes_list
bottom5_genes = HNscore_add_metadata_with_score.join(
    bottom5_genes_list.select("transcript_id"),
    on="transcript_id",
    how="semi" # Returns rows from the left table that have a match in the right table.
)

bottom5_genes.write_csv(
    f"../../Data/HNscore/HNscore_transcript/HNscore_gencode_{VERSION}/annotation/HN{threshold}_bottom5p_transcripts_annotation_{Execute_date}.tsv",
    separator="\t"
)

display(bottom5_genes)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,u32,str,str,str,str,str,u32,str,str
"""ENST00000535641.5""",84,132,190,-48,58,64,129,-6,"""ANAPC5""","""HGNC:15713""","""retained_intron""","""ENSG00000089053.14""","""""","""-""",0,null,null,"""polya_signal, polya_site""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000834029.1""",69,117,220,-48,47,56,148,-9,"""EXO5-DT""","""HGNC:55647""","""lncRNA""","""ENSG00000238287.2""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A1647B""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000943010.1""",50,98,258,-48,26,35,190,-9,"""MINDY1""","""HGNC:25648""","""protein_coding""","""ENSG00000143409.19""","""basic, TAGENE""","""-""",2,null,null,"""polya_signal, polya_site""","""ensembl_havana_gene_homo_sapiens""","""ENCODE_tissue""",1,"""Other""","""protein_coding"""
"""ENST00000510365.5""",64,112,230,-48,44,53,154,-9,"""ACOX3""","""HGNC:121""","""retained_intron""","""ENSG00000087008.17""","""""","""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000829178.1""",85,133,188,-48,59,68,124,-9,null,null,"""lncRNA""","""ENSG00000291153.2""","""TAGENE""","""+""",1,null,null,null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000497231.7""",15,266,125,-251,10,182,59,-172,"""ALDH7A1""","""HGNC:877""","""retained_intron""","""ENSG00000164904.19""","""""","""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000490762.5""",12,265,129,-253,10,174,67,-164,"""LUC7L""","""HGNC:6723""","""retained_intron""","""ENSG00000007392.18""","""""","""-""",0,null,null,"""polya_site, polya_signal""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""
"""ENST00000426094.5""",11,265,130,-254,11,165,75,-154,"""LUC7L""","""HGNC:6723""","""nonsense_mediated_decay""","""ENSG00000007392.18""","""""","""-""",0,null,null,"""polya_site, polya_signal""","""ensembl_havana_gene_homo_sapiens""",null,0,"""no annotation remark""","""others"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Pie Chart (transcript category)

In [22]:
# Define custom colors for each gene category
transcript_category_colors = {
    'protein_coding': '#EF553B',  # Red-orange (画像のmRNAに合わせて赤みがかったオレンジ色)
    'others': '#D3D3D3',  # LightGray (薄い灰色)
    'lncRNA': '#AB63FA'  # Purple (画像のDivergent p-lncRNAに合わせて紫色)
    # 'p-lncRNA, intergenic': '#4682B4',  # Steelblue (画像のIntergenic p-lncRNAに合わせてsteelblue)
    # 'e-lncRNA': '#00CC96'  # Teal-green (画像のe-lncRNAに合わせて緑色)
}

transcript_category_counts_up = (
    top5_genes
    .group_by("representative_transcript_type")  # Group by gene category column
    .agg(pl.len().alias("count"))  # Count occurrences
    .sort("count", descending=True)  # Sort by count in descending order
)

# Enhanced pie chart with custom colors
fig_basic = px.pie(
    transcript_category_counts_up.to_pandas(),  # Convert Polars to Pandas for plotly
    values='count',
    names='representative_transcript_type',
    title='Transcript Type Distribution (Upregulated genes)',
    color='representative_transcript_type',  # Add color parameter
    color_discrete_map=transcript_category_colors  # Apply custom color mapping
)

# Customize the appearance
fig_basic.update_traces(
    textposition='outside',  # Place text outside pie slices
    textinfo='value+percent+label',  # Show value, percentage and label
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

# Update layout for better appearance
fig_basic.update_layout(
    font_size=18,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.01
    )
)

fig_basic.show()

In [23]:
# Define custom colors for each gene category
transcript_category_colors = {
    'protein_coding': '#EF553B',  # Red-orange (画像のmRNAに合わせて赤みがかったオレンジ色)
    'others': '#D3D3D3',  # LightGray (薄い灰色)
    'lncRNA': '#AB63FA'  # Purple (画像のDivergent p-lncRNAに合わせて紫色)
    # 'p-lncRNA, intergenic': '#4682B4',  # Steelblue (画像のIntergenic p-lncRNAに合わせてsteelblue)
    # 'e-lncRNA': '#00CC96'  # Teal-green (画像のe-lncRNAに合わせて緑色)
}

transcript_category_counts_down = (
    bottom5_genes
    .group_by("representative_transcript_type")  # Group by gene category column
    .agg(pl.len().alias("count"))  # Count occurrences
    .sort("count", descending=True)  # Sort by count in descending order
)

# Enhanced pie chart with custom colors
fig_basic = px.pie(
    transcript_category_counts_down.to_pandas(),  # Convert Polars to Pandas for plotly
    values='count',
    names='representative_transcript_type',
    title='Transcript Type Distribution (Downregulated genes)',
    color='representative_transcript_type',  # Add color parameter
    color_discrete_map=transcript_category_colors  # Apply custom color mapping
)

# Customize the appearance
fig_basic.update_traces(
    textposition='outside',  # Place text outside pie slices
    textinfo='value+percent+label',  # Show value, percentage and label
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

# Update layout for better appearance
fig_basic.update_layout(
    font_size=18,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.01
    )
)

fig_basic.show()

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Pie Chart (lncRNA annotation remarks)

In [24]:
top5_genes_lncRNA = top5_genes.filter(
    pl.col("representative_transcript_type") == "lncRNA"
)

print(f"top5 lncRNA transcript count: {top5_genes_lncRNA.select('transcript_id').n_unique()}")
display(top5_genes_lncRNA)

top5 lncRNA transcript count: 4716


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,u32,str,str,str,str,str,u32,str,str
"""ENST00000850140.1""",317,6,83,311,207,2,42,205,null,null,"""lncRNA""","""ENSG00000310471.1""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""+""",4,"""URS0002A15385""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project, overlaps pseudogene""",2,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000738043.1""",312,9,85,303,195,4,52,191,null,null,"""lncRNA""","""ENSG00000296306.1""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000729765.1""",315,20,71,295,206,4,41,202,null,null,"""lncRNA""","""ENSG00000295391.1""","""basic, TAGENE""","""+""",2,"""URS0002A422B0""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000778743.1""",306,17,83,289,207,2,42,205,null,null,"""lncRNA""","""ENSG00000301409.1""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""+""",4,"""URS0002A31C49""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000570843.1""",303,19,84,284,201,8,42,193,null,null,"""lncRNA""","""ENSG00000261889.1""","""basic, Ensembl_canonical, GENCODE_Primary""","""-""",3,"""URS00003E2626""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,"""no annotation remark""","""lncRNA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000789046.1""",87,43,276,44,27,21,203,6,null,null,"""lncRNA""","""ENSG00000302714.1""","""TAGENE""","""-""",1,"""URS0002A34B31""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000547282.2""",60,16,330,44,7,2,242,5,null,null,"""lncRNA""","""ENSG00000257221.5""","""basic, TAGENE""","""+""",2,"""URS0002A236B4""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000808863.1""",52,8,346,44,7,3,241,4,null,null,"""lncRNA""","""ENSG00000305118.1""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A622FF""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""


In [25]:
bottom5_genes_lncRNA = bottom5_genes.filter(
    pl.col("representative_transcript_type") == "lncRNA"
)

print(f"bottom5 lncRNA transcript count: {bottom5_genes_lncRNA.select('transcript_id').n_unique()}")
display(bottom5_genes_lncRNA)

bottom5 lncRNA transcript count: 3223


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,u32,str,str,str,str,str,u32,str,str
"""ENST00000834029.1""",69,117,220,-48,47,56,148,-9,"""EXO5-DT""","""HGNC:55647""","""lncRNA""","""ENSG00000238287.2""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A1647B""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000829178.1""",85,133,188,-48,59,68,124,-9,null,null,"""lncRNA""","""ENSG00000291153.2""","""TAGENE""","""+""",1,null,null,null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000524304.1""",27,75,304,-48,18,29,204,-11,"""HOXA-AS3""","""HGNC:43748""","""lncRNA""","""ENSG00000254369.7""","""basic""","""+""",1,"""URS000007890C""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,"""no annotation remark""","""lncRNA"""
"""ENST00000739605.1""",68,116,222,-48,46,57,148,-11,null,null,"""lncRNA""","""ENSG00000296428.1""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A51155""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000730725.1""",45,93,268,-48,23,35,193,-12,null,null,"""lncRNA""","""ENSG00000285728.4""","""basic, TAGENE""","""-""",2,"""URS0002A6FD9A""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000803049.1""",35,227,144,-192,12,152,87,-140,null,null,"""lncRNA""","""ENSG00000278903.6""","""TAGENE""","""+""",1,"""URS0002A3AB0D""","""ENSEMBL_GENCODE""",null,"""ensembl_havana_lincrna_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000692499.2""",12,207,187,-195,6,140,105,-134,null,null,"""lncRNA""","""ENSG00000289378.2""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""-""",4,"""URS0002A18CC9""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""Assembled from PacBio CLS3 reads, CLS3 project""",2,"""derived from CLS3 project""","""lncRNA"""
"""ENST00000619697.3""",11,209,186,-198,9,142,100,-133,null,null,"""lncRNA""","""ENSG00000278834.3""","""basic, Ensembl_canonical, GENCODE_Primary, TAGENE""","""+""",4,"""URS0002A54E6E""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","""CLS3 project, Assembled from PacBio CLS3 reads""",2,"""derived from CLS3 project""","""lncRNA"""


In [26]:
# Define custom colors for each gene category
annotation_remark_colors = {
    'MANE_select': '#EF553B',  # Red-orange (画像のmRNAに合わせて赤みがかったオレンジ色)
    'no annotation remark': '#D3D3D3',  # LightGray (薄い灰色)
    'derived from CLS3 project': '#AB63FA',  # Purple (画像のDivergent p-lncRNAに合わせて紫色)
    'derived from LRGASP': '#4682B4',  # Steelblue (画像のIntergenic p-lncRNAに合わせてsteelblue)
    'Other': '#00CC96'  # Teal-green (画像のe-lncRNAに合わせて緑色)
}

transcript_category_counts_up = (
    top5_genes_lncRNA
    .group_by("representative_annotation_remarks")  # Group by gene category column
    .agg(pl.len().alias("count"))  # Count occurrences
    .sort("count", descending=True)  # Sort by count in descending order
)

# Enhanced pie chart with custom colors
fig_basic = px.pie(
    transcript_category_counts_up.to_pandas(),  # Convert Polars to Pandas for plotly
    values='count',
    names='representative_annotation_remarks',
    title='LncRNA Representative Annotation Remarks Distribution (Upregulated genes)',
    color='representative_annotation_remarks',  # Add color parameter
    color_discrete_map=annotation_remark_colors  # Apply custom color mapping
)

# Customize the appearance
fig_basic.update_traces(
    textposition='outside',  # Place text outside pie slices
    textinfo='value+percent+label',  # Show value, percentage and label
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

# Update layout for better appearance
fig_basic.update_layout(
    font_size=18,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.01
    )
)

fig_basic.show()

In [27]:
# Define custom colors for each gene category
annotation_remark_colors = {
    'MANE_select': '#EF553B',  # Red-orange (画像のmRNAに合わせて赤みがかったオレンジ色)
    'no annotation remark': '#D3D3D3',  # LightGray (薄い灰色)
    'derived from CLS3 project': '#AB63FA',  # Purple (画像のDivergent p-lncRNAに合わせて紫色)
    'derived from LRGASP': '#4682B4',  # Steelblue (画像のIntergenic p-lncRNAに合わせてsteelblue)
    'Other': '#00CC96'  # Teal-green (画像のe-lncRNAに合わせて緑色)
}

transcript_category_counts_down = (
    bottom5_genes_lncRNA
    .group_by("representative_annotation_remarks")  # Group by gene category column
    .agg(pl.len().alias("count"))  # Count occurrences
    .sort("count", descending=True)  # Sort by count in descending order
)

# Enhanced pie chart with custom colors
fig_basic = px.pie(
    transcript_category_counts_down.to_pandas(),  # Convert Polars to Pandas for plotly
    values='count',
    names='representative_annotation_remarks',
    title='LncRNA Representative Annotation Remarks Distribution (Downregulated genes)',
    color='representative_annotation_remarks',  # Add color parameter
    color_discrete_map=annotation_remark_colors  # Apply custom color mapping
)

# Customize the appearance
fig_basic.update_traces(
    textposition='outside',  # Place text outside pie slices
    textinfo='value+percent+label',  # Show value, percentage and label
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)

# Update layout for better appearance
fig_basic.update_layout(
    font_size=18,
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.01
    )
)

fig_basic.show()